### Heart Disease Prediction Using Machine Learning Algorithms from Scratch

This project focuses on predicting the presence and severity of heart disease using a real-world medical dataset from UCI ML Repository. The dataset contains patient-related medical and demographic information, such as age, sex, chest pain type, blood pressure, cholesterol levels, and other clinical measurements. The target variable, `goal`, indicates whether heart disease is present in a patient.It is multivariate and contains 14 attributes. The original target variable is multiclass, with five possible values:

0: absence of heart disease
1, 2, 3, 4: presence of heart disease with different levels of severity

Dataset available at __[UCI repository](https://archive.ics.uci.edu/dataset/45/heart+disease)__  


**All column names listed:**


`age`: age in years       

`sex`: sex (1 = male; 0 = female)       

`cp`: chest pain type
        -- Value 1: typical angina
        -- Value 2: atypical angina
        -- Value 3: non-anginal pain
        -- Value 4: asymptomatic        
        
`trestbps`: resting blood pressure (in mm Hg on admission to the hospital)  

`chol`: serum cholestoral in mg/dl      

`fbs`: (fasting blood sugar > 120 mg/dl)  (1 = true; 0 = false)   

`restecg`: resting electrocardiographic results
        -- Value 0: normal
        -- Value 1: having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV)
        -- Value 2: showing probable or definite left ventricular hypertrophy by Estes' criteria   
        
`thalach`: maximum heart rate achieved  

`exang`: exercise induced angina (1 = yes; 0 = no)

`oldpeak`: ST depression induced by exercise relative to rest   

`slope`: the slope of the peak exercise ST segment
        -- Value 1: upsloping
        -- Value 2: flat
        -- Value 3: downsloping    

`ca`: number of major vessels (0-3) colored by flourosopy        

`thal`: 3 = normal; 6 = fixed defect; 7 = reversable defect      

`num`       (the predicted attribute)

**Imports**

In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
import plotly.express as px

In [2]:
# %pip install --force-reinstall "plotly==5.24.1"

**Importing and Preparing the dataset**

In [3]:
# pip install ucimlrepo

In [4]:
def fetch_dataset():
# fetching dataset
    heart_disease = fetch_ucirepo(id=45)

    X = heart_disease.data.features
    y = heart_disease.data.targets
    # concatenate (X) and targets (y) into a single dataframe
    heart_disease_df = pd.concat([X, y], axis=1)

    return heart_disease_df

# original table view
original_dataset = fetch_dataset()
heart_disease_df = original_dataset
print("Original dataset:")
display(heart_disease_df)

Original dataset:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0,1


In [5]:
# Dropping null values
def drop_NAN(dataset):
    return dataset.dropna().copy()

def drop_Target(dataset):
    return dataset.drop(columns=['num'], errors='ignore')

heart_disease_df = drop_NAN(heart_disease_df)
heart_disease_df = drop_Target(heart_disease_df)
print("Clean dataset:")
display(heart_disease_df)

Clean dataset:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,57,0,4,140,241,0,0,123,1,0.2,2,0.0,7.0
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0


**Helper functions**

In [6]:
def getTargets(df):
    return df.iloc[:, -1]   # The last column as the target

def getDfnoTargets(df):
    return df.iloc[:, :-1]  # All columns except the last one as features

In [7]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

In [8]:
def convert_target(dataframe):
    dataframe.loc[:, 'num'] = dataframe['num'].apply(lambda x: 1 if x > 0 else 0)
    return dataframe

In [9]:
def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

### Part 1: Principal Component Analysis (PCA)

We want to find the direction of the first principal component, such that the orthogonal projection x' on u (i.e variance) of the data points x is maximum.
This translates to an eigenvector/ eigenvalue problem.

$$
\max_{\mathbf{u}} \; \mathbf{u}^\top \mathbf{S} \mathbf{u}
\quad \text{given constraint} \quad
\mathbf{u}^\top \mathbf{u} = 1
$$

**Dealing with categorical variables before PCA**

the categorical features like *sex (1 = male; 0 = female)* are already represented in binary, thus neither conversion nor normalization will be needed for these.

Categorical variables like *cp(1: typical angina, 2: atypical angina,3: non-anginal pain 4: asymptomatic)* will be converted to one hot vectors which will be inserted back to the table as features

In [10]:
categorical_columns = ['restecg', 'slope', 'cp', 'thal']

def one_hot_vector(data, column):
    # fetching distinct values
    range_of_values = data[column].unique()
    for value in range_of_values:
        data[f"{column}_{value}"] = (data[column] == value).astype(int)
    data = data.drop(columns=[column])

for item in categorical_columns:
    one_hot_vector(heart_disease_df, item)

print("Dataframe with one hot vector encoding:")
display(heart_disease_df)

Dataframe with one hot vector encoding:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,...,slope_3,slope_2,slope_1,cp_1,cp_4,cp_3,cp_2,thal_6.0,thal_3.0,thal_7.0
0,63,1,1,145,233,1,2,150,0,2.3,...,1,0,0,1,0,0,0,1,0,0
1,67,1,4,160,286,0,2,108,1,1.5,...,0,1,0,0,1,0,0,0,1,0
2,67,1,4,120,229,0,2,129,1,2.6,...,0,1,0,0,1,0,0,0,0,1
3,37,1,3,130,250,0,0,187,0,3.5,...,1,0,0,0,0,1,0,0,1,0
4,41,0,2,130,204,0,2,172,0,1.4,...,0,0,1,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,57,0,4,140,241,0,0,123,1,0.2,...,0,1,0,0,1,0,0,0,0,1
298,45,1,1,110,264,0,0,132,0,1.2,...,0,1,0,1,0,0,0,0,0,1
299,68,1,4,144,193,1,0,141,0,3.4,...,0,1,0,0,1,0,0,0,0,1
300,57,1,4,130,131,0,0,115,1,1.2,...,0,1,0,0,1,0,0,0,0,1


In [11]:
# performing normalization
numerical_columns = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

def normalization(numerical_columns, dataframe):
    for column in numerical_columns:
        # Normalize by subtracting mean and dividing by standard deviation
        dataframe[column] = (dataframe[column] - dataframe[column].mean()) / dataframe[column].std()

normalization(numerical_columns, heart_disease_df)
print("Dataframe after Normalization:")
display(heart_disease_df)

Dataframe after Normalization:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,...,slope_3,slope_2,slope_1,cp_1,cp_4,cp_3,cp_2,thal_6.0,thal_3.0,thal_7.0
0,0.934603,1,1,0.749116,-0.275978,1,2,0.017465,0,1.067164,...,1,0,0,1,0,0,0,1,0,0
1,1.376605,1,4,1.593577,0.743301,0,2,-1.813274,1,0.381130,...,0,1,0,0,1,0,0,0,1,0
2,1.376605,1,4,-0.658320,-0.352904,0,2,-0.897904,1,1.324427,...,0,1,0,0,1,0,0,0,0,1
3,-1.938409,1,3,-0.095345,0.050961,0,0,1.630258,0,2.096215,...,1,0,0,0,0,1,0,0,1,0
4,-1.496407,0,2,-0.095345,-0.833696,0,2,0.976423,0,0.295376,...,0,0,1,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,0.271600,0,4,0.467629,-0.122124,0,0,-1.159438,1,-0.733675,...,0,1,0,0,1,0,0,0,0,1
298,-1.054405,1,1,-1.221294,0.320204,0,0,-0.767137,0,0.123867,...,0,1,0,1,0,0,0,0,0,1
299,1.487106,1,4,0.692818,-1.045244,1,0,-0.374836,0,2.010461,...,0,1,0,0,1,0,0,0,0,1
300,0.271600,1,4,-0.095345,-2.237607,0,0,-1.508150,1,0.123867,...,0,1,0,0,1,0,0,0,0,1


***Computing Covariance Matrix (S)***

In [12]:
# Convert the DataFrame to a numpy array
heart_disease_array = heart_disease_df.to_numpy()

# Calculate the covariance matrix using the provided formula
num_rows = heart_disease_array.shape[0]  # Number of rows in the data
covariance_matrix = (1 / num_rows) * np.dot(heart_disease_array.T, heart_disease_array)

# Convert the covariance matrix to a DataFrame for readability
covariance_matrix_df = pd.DataFrame(
    covariance_matrix,
    index=heart_disease_df.columns,
    columns=heart_disease_df.columns
)

#display(covariance_matrix_df)  #whole covariance matrix

# checking how covariance matrix looks
cov_subset = covariance_matrix_df.iloc[:10, :10]
print("First 10 rows and columns of Covariance Matrix")
display(cov_subset)

First 10 rows and columns of Covariance Matrix


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak
age,0.996633,-0.043143,0.106230,0.289498,0.201961,0.046392,0.148652,-0.393234,0.045174,0.196459
sex,-0.043143,0.676768,2.141414,-0.030976,-0.092492,0.104377,0.690236,-0.028247,0.252525,0.049759
cp,0.106230,2.141414,10.902357,-0.035560,0.069321,0.437710,3.208754,-0.326282,1.202020,0.195441
trestbps,0.289498,-0.030976,-0.035560,0.996633,0.131093,0.063534,0.147983,-0.048942,0.031223,0.190599
chol,0.201961,-0.092492,0.069321,0.131093,0.996633,0.004464,0.163654,-0.000074,0.027781,0.038466
fbs,0.046392,0.104377,0.437710,0.063534,0.004464,0.144781,0.168350,-0.002755,0.047138,0.002919
restecg,0.148652,0.690236,3.208754,0.147983,0.163654,0.168350,1.979798,-0.071680,0.363636,0.112767
thalach,-0.393234,-0.028247,-0.326282,-0.048942,-0.000074,-0.002755,-0.071680,0.996633,-0.179953,-0.346469
exang,0.045174,0.252525,1.202020,0.031223,0.027781,0.047138,0.363636,-0.179953,0.326599,0.135449
oldpeak,0.196459,0.049759,0.195441,0.190599,0.038466,0.002919,0.112767,-0.346469,0.135449,0.996633


**Eigenvectors of the Covariance matrix:** are actually the directions of the axes where there is the most variance, meaning most information (Principal Components).

**Eigenvalues:** are the coefficients attached to eigenvectors, which give the amount of variance carried in each Principal Component.

For a high-dimensional dataset like this one, eigenvectors define a new rotated coordinate system where:

The first axis (eigenvector) is along the maximum variance direction.
The second axis (eigenvector) is perpendicular and captures the next highest variance, and so on.
This is how (PCA) works, by rotating the coordinate system so that data is most spread along the first few axes.

Finding and sorting eigenvalues and eigenvestors, by ranking your eigenvectors in order of their eigenvalues, highest to lowest, you get the principal components in order of significance.

***Eigenvalue Decomposition: solving SV=VΛ where Λ is the diagonal***

In [13]:
def eigsort(cov_matrix):
    # find eigenvalues and eigenvectors of the covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov_matrix)
    sorted_indices = np.argsort(eigvals)[::-1]
    sorted_eigenvalues = eigvals[sorted_indices]          # Sort eigenvalues
    sorted_eigenvectors = eigvecs[:, sorted_indices]      # Sort eigenvectors
    return sorted_eigenvectors, sorted_eigenvalues


def decomposition(dataframe, cov_matrix, k):
    # Calculate eigenvalues and eigenvectors
    eigvectors, eigvals = eigsort(cov_matrix)

    # Select the top M eigenvectors
    U = eigvectors[:, :k]

    # Select the top M eigenvalues
    Lambdas = eigvals[:k]

    # Project the data onto the top M components
    Z = np.dot(dataframe, U)

    return Z, U, Lambdas

Decomposition is used to select the k components with the highest variance, in other words the principle compoents with the most information.

1st Principal Component (PC1): The direction of maximum variance (most spread).
2nd Principal Component (PC2): The next best direction, perpendicular to PC1

In [14]:
#applying PCA

def pca_classic(X, k, covariance_matrix):
#     X (ndarray): Preprocessed dataset (already normalized).
#     k (int): Number of principal components to retain.
#     covariance_matrix (ndarray): Precomputed covariance matrix.

#     Z (ndarray): Projected dataset onto k principal components.
#     U (ndarray): Selected eigenvectors (principal components).
#     Lambdas (ndarray): Corresponding eigenvalues.
#     mu (ndarray): Mean vector used for normalization.

    # mean vector
    mu = X.mean(axis=0).reshape(1, -1)

    # PCA decomposition (eigenvalue & eigenvector extraction)
    Z, U, Lambdas = decomposition(X, covariance_matrix, k)

    return Z, U, Lambdas, mu

# num of Principal components to retain
M = 8

# PCA using covariance matrix
Z, eigvecs, eigvals, mu = pca_classic(heart_disease_array, M, covariance_matrix)

In [15]:
# const5ructing a dataset with z values
pca_columns = [f"PC{i + 1}" for i in range(Z.shape[1])]# number of columns

pca_df = pd.DataFrame(Z,columns=pca_columns)

pca_df["label"] = getTargets(original_dataset)

fig = px.parallel_coordinates(
    pca_df,
    dimensions=["label"] + pca_columns,
    color="label",
    color_continuous_scale=px.colors.sequential.Bluered,
    title="Parallel Coordinates Plot of PCA Components",
    labels={
        "PC1": "PC1",
        "PC2": "PC2",
        "PC3": "PC3",
        "PC4": "PC4",
        "PC5": "PC5",
        "label": "Disease Class"
    }
)

# print(type(fig))
# print(pca_df.shape)
# print(pca_df.head())
# import notebook
# print(notebook.__version__)
fig.show()

The first PCA has wider distribution of scores which reflects the fact that the 1st component should capture most of the variance. In contrast, the later principal components display a narrower spread of values, indicating that they explain progressively smaller amounts of variance (information). Although the principal components are orthogonal (uncorrelated), the zig-zag patterns arise since an observation that projects strongly positively onto PC1 may project negatively onto PC2 simply because PC2 measures a different direction.

**Visualing Reconstructed Data**

In [16]:
# reconstruct the data from PCA components
X_rec = Z.dot(eigvecs.T) + mu

# randomize permutationz of dataset indices for visualization
ind = np.random.permutation(heart_disease_array.shape[0])

In [17]:
def style_figure(fig, title, x_title, y_title, height=450):
    fig.update_layout(title={"text": title, "x": 0.5, "xanchor": "center"}, xaxis_title=x_title, yaxis_title=y_title,
                      template="plotly_white", height=height, hovermode="closest",
                      font=dict(family="Arial", size=13, color="#2B2B2B"), legend_title_text="")
    fig.update_xaxes(showgrid=True, gridcolor="#E5E7EB")
    fig.update_yaxes(showgrid=True, gridcolor="#E5E7EB")
    return fig

df_tmp = pd.DataFrame({"Feature 1": np.concatenate([np.atleast_1d(heart_disease_array[ind, 0]), np.atleast_1d(X_rec[ind, 0])]),
                       "Feature 2": np.concatenate([np.atleast_1d(heart_disease_array[ind, 1]), np.atleast_1d(X_rec[ind, 1])]),
"Dataset": ["Original Data"] * len(np.atleast_1d(heart_disease_array[ind, 0])) + ["Reconstructed Data"] * len(np.atleast_1d(X_rec[ind, 0]))})

fig = px.scatter(df_tmp, x="Feature 1", y="Feature 2", color="Dataset", symbol="Dataset",
                 color_discrete_map={"Original Data": "#2563EB", "Reconstructed Data": "#F97316"}, opacity=0.7)
fig.update_traces(marker=dict(size=8))
style_figure(fig, "Original vs Reconstructed Features Using PCA", "Feature 1", "Feature 2")
fig.show()

**Evaluation of PCA:** the first two normalized features of the reconstructed dataset are compared against their corresponding values in the original dataset. The second feature is binary `sex`, while the first feature `age` is continuous and has been normalized prior to PCA.

The reconstructed observations do not perfectly overlap with the original observations, indicating that some information was lost during dimensionality reduction. This loss is expected, as PCA retains only a subset of the principal components and discards directions associated with lower variance.

### Part 2: Least Squares Algorithm for classification task

In order for Least Square algorithm will be used for classification task, the problem will be reformulated as follows: having a linear equation ***y = θ0 + θ1x1+...+θmxm*** and some data points the goal is to minimize the least squares error between X data matrix times vector of theta values, minus the predicted values, which are known.

***The objective function is:***
$$
E(\theta) = \left\|X\theta - y\right\|^2
$$

Each point X is a vector of features of a row, accompanied by its corresponding target value

The goal is to obtain values for **θ = (θ0, θ1, . . . , θd )** that result
in the best fit.

***The optimal parameters are obtained by solving the Normal Equation***

$$
X^{T}X\theta=X^{T}y
$$

***Which yields the closed-form solution***

$$
\theta=(X^{T}X)^{-1}X^{T}y.
$$

<div class=" alert alert-block alert-info">
    <b> target column has values 0-3, it will be interpreted as numerical range by the algorithm, therefore the num column will be encoded with one hot vector</b>
</div>

<div class=" alert alert-block alert-info">
    <b> Since a system of linear equations has to be solved, the normalized version of dataset will be used, so that features with larger numerical scales do not disproportionately influence the solution.
</b>
</div>

In [18]:
X = heart_disease_df

temp_df = original_dataset
temp_df_clean = drop_NAN(temp_df)
# encoding the target column for this task
one_hot_vector(temp_df_clean, "num")
display(temp_df_clean)
num_cols = [c for c in temp_df_clean.columns if c.startswith("num_")]
y = temp_df_clean[num_cols].to_numpy(dtype=float)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,num_0,num_2,num_1,num_3,num_4
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0,1,0,0,0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2,0,1,0,0,0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1,0,0,1,0,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0,1,0,0,0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,57,0,4,140,241,0,0,123,1,0.2,2,0.0,7.0,1,0,0,1,0,0
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1,0,0,1,0,0
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2,0,1,0,0,0
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3,0,0,0,1,0


 <div class=" alert alert-block alert-info">
    <b> Note: in the initial X matrix we will add a bias column y-intercept (θ0) of value 1. This will allow the value to change as we search for the optimal opposing to having no valuue at all(meaning as y is 0 X would be 0 as well)
</b>
</div>

 <div class=" alert alert-block alert-info">
    <b>Instead of X, Xtilde will be used, which is the augmented input vector.The original feature vector with a 1 prepended to handle the bias (intercept) term.
</b>
</div>

In [19]:
def least_squares(X,y):
    Num_of_rows = X.shape[0]
    # adding a bias column (y-intercept) of
    # X has the shape:
    #   [x11, x12, x13, ..., x1m],
    #   [x21, x22, x23, ..., x2m],
    #   ...
    #   [xn1, xn2, xn3, ..., xnm]
    X_tilde = np.c_[np.ones((Num_of_rows, 1)), X]
    #print(X_tilde)

    LHS = X_tilde.T @ X_tilde
    RHS = X_tilde.T @ y

    theta = np.linalg.solve(LHS, RHS) # Step B: solve LHSw=RHS
    #print(theta)

    scores = X_tilde @ theta

    return scores

In [20]:
ls_scores = least_squares(X,y)

y_prediction = np.argmax(ls_scores, axis=1)

y_true = np.argmax(y, axis=1)
accuracy = accuracy_score(y_true, y_prediction)
print(f"Accuracy of Least Squares: {accuracy * 100 :.2f} %")

Accuracy of Least Squares: 63.64 %


### Part 3 : Logistic regression using stochastic gradient descent ###

Implementation of simple Logistic Regression. For an observation "x", it models the probability that the observation belongs to the positive class. First, a linear model of features is computed:

$$
z = \theta^{T}x
$$

where (θ) is the vector of model parameters.
They are later passed to a **sigmoid function**, which outputs probablities ranging from 0-1.


A bias column of ones is added to the feature matrix, allowing the model to learn an intercept term, enabling the decision boundary to shift independently of the feature values.

Rather than minimizing squared error, Logistic Regression seeks the vector of params that minimizes the **Cross-entropy loss**.
This loss function heavily penalizes confident but incorrect predictions and is therefore suited for classification problems.

Since no closed form solution exists for minimizing the Cross-entropy, the model parameters are estimated iteratively using Stochastic Gradient Descent (SGD) technique


This is a binary implementation of logistic regression, the sigmoid outputs 0<y<1, and this probability is interpreted as:

$$
P(y=1\mid x)=\hat{y},
$$

while the probability of the negative class is

$$
P(y=0\mid x)=1-\hat{y}.
$$

In [21]:
# Preparing and splitting the dtaset
lg_df_clean = drop_NAN(original_dataset)
lg_df = convert_target(lg_df_clean)
X = getDfnoTargets(heart_disease_df)
y = getTargets(lg_df)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Set Shape (X_train, y_train):", X_train.shape, y_train.shape)
print("Test Set Shape (X_test, y_test):", X_test.shape, y_test.shape)

Training Set Shape (X_train, y_train): (237, 25) (237,)
Test Set Shape (X_test, y_test): (60, 25) (60,)


In [22]:
#Convert to numpy arrays
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values

# Adding a bias column
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]

# sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Cross-entropy loss
def cross_entropy_loss(y, y_pred):
    return -np.mean(y * np.log(y_pred + 1e-15) + (1 - y) * np.log(1 - y_pred + 1e-15))

# Stochastic Gradient Descent for Logistic Regression
# using sigmoid

def train_sgd(X, y, learning_rate=0.01, epochs=100):
    weights = np.zeros(X.shape[1])  # Initialize weights
    losses = []
    for epoch in range(epochs):
        for i in range(X.shape[0]):
            idx = np.random.randint(X.shape[0])
            xi = X[idx]
            yi = y[idx]

            y_pred = sigmoid(np.dot(xi, weights))
            gradient = (y_pred - yi) * xi

            weights -= learning_rate * gradient

        # Calculating loss
        y_preds = sigmoid(np.dot(X, weights))
        loss = cross_entropy_loss(y, y_preds)
        losses.append(loss)
        #print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss}")

    return weights, losses


# Running the sigmoid function with established weights
def predict(X, weights):
    y_pred = sigmoid(np.dot(X, weights))
    return [1 if i > 0.5 else 0 for i in y_pred]

#training model
weights, losses = train_sgd(X_train, y_train, learning_rate=0.01, epochs=100)

# predicting using sigmoid and weights
y_pred_test = predict(X_test, weights)
y_pred_train = predict(X_train, weights)

# Accuracy
test_accuracy = accuracy_score(y_test, y_pred_test)
train_accuracy = accuracy_score(y_train, y_pred_train)

print(f"Train Accuracy of LR: {train_accuracy* 100 :.2f} %")
print(f"Test Accuracy of LR: {test_accuracy * 100 :.2f} %")

Train Accuracy of LR: 86.92 %
Test Accuracy of LR: 86.67 %


In [23]:
import plotly.graph_objects as go
def plot_learning_curve(losses, title="Training Learning Curve"):
    epochs = np.arange(1, len(losses) + 1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=epochs, y=losses, mode="lines", name="Training loss", line=dict(color="#2563EB", width=3),
                             hovertemplate="Epoch: %{x}<br>Loss: %{y:.4f}<extra></extra>"))
    style_figure(fig, title, "Epoch", "Cross-Entropy Loss")
    fig.show()

plot_learning_curve(losses)

Training accuracy and test accuracy are quite high, demonstrating a strong predictive performance on both the training and unseen data. The similarity between the training and test accuracies indicates that the model generalizes well and does not exhibit significant overfitting.

The learning curve shows that the cross-entropy loss decreases rapidly during the first epochs, indicating that the model quickly learns useful parameter values.The fluctuations are normal because SGD updates the weights using randomly selected training examples, which introduces noise into the optimization process. Overall, the downward trend confirms that this Logistic Regression model is successfully minimizing the training loss.

### Part 4 : K-Nearest Neighbors Algorithm ###

Unlike Logistic Regression or Least Squares Classification, KNN does not learn an explicit model during training. Instead, predictions are made by comparing a new observation to previously observed training instances.

**Eucledian distance** will be used, it will be computed between parameter x and every training observation x_i

The K observations with the smallest distances are then selected as the nearest neighbors. Each neighbor contributes one vote for its class label, and the class receiving the majority of votes is assigned to the query observation.

The goal is to find the optimal value of the **hyperparameter K in the interval [1, 10]**, where K is the number of neighbors to be considered.

The error function, will be K-fold cross validation

 <div class=" alert alert-block alert-info">
    <b>For this task the target column values will be substituted with 0 for absence and 1 for presence.
</b>
</div>

In [24]:
from sklearn.preprocessing import MinMaxScaler

knn_df_clean = drop_NAN(original_dataset)

# modifying num column
knn_df = convert_target(knn_df_clean)

X = getDfnoTargets(knn_df)
y = getTargets(knn_df)

# scaling the data so that the magnitude of certain values dont affect the distance
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [25]:
def square_distance(x,y):
    y = np.tile(y,(x.shape[0],1))
    distance = np.sqrt((np.square(x-y).sum(axis=1)))
    return distance

def knn_algorithm(x_train, y_train, x_test, K):
    # K is the number of close neighbors to consider
    Num_xtest = x_test.shape[0]

    Ttest = np.zeros(Num_xtest, dtype=int)
    for element in range(Num_xtest):
        # calculating square distance for each test instance in test data
        d = square_distance(x_train, x_test[element, :])
        # find k nearest neighbors
        nearest_n = np.argsort(d)[:K]
        labels_for_n = y_train[nearest_n]
        votes = np.bincount(labels_for_n)
        Ttest[element] = np.argmax(votes)  # find the most voted class
        #print(Ttest)# 1d array
    return Ttest

In [26]:
def cross_validation(K, numFolds, X_train, y_train):
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    N = X_train.shape[0]

    np.random.seed(10)
    perm = np.random.permutation(N)
    X_train = X_train[perm]
    y_train = y_train[perm]

    Xfolds = np.array_split(X_train, numFolds)
    Tfolds = np.array_split(y_train, numFolds)
    val_err_values = np.zeros((K, numFolds))

    for k in range(1, K + 1):
        for j in range(numFolds):
            X_fold_train = np.vstack(Xfolds[:j] + Xfolds[j + 1:])
            y_fold_train = np.concatenate(Tfolds[:j] + Tfolds[j + 1:])
            X_fold_test = Xfolds[j]
            y_fold_test = Tfolds[j]

            y_pred = knn_algorithm(X_fold_train, y_fold_train, X_fold_test, k)
            val_err_values[k - 1, j] = np.mean(y_pred != y_fold_test)

    val = val_err_values.mean(axis=1)
    kbest = np.argmin(val) + 1

    return kbest, val, val_err_values

kbest, val, val_err_values = cross_validation(10, 5, X_train, y_train)

In [27]:
validation_errors = np.asarray(val, dtype=float)
k_values = np.arange(1, 10 + 1)
best_index = np.argmin(validation_errors)
kbest = int(k_values[best_index])
best_error = validation_errors[best_index]
fig = go.Figure()
fig.add_trace(go.Scatter(x=k_values, y=validation_errors * 100, mode="lines+markers", name="Validation error",
                         line=dict(color="#2563EB", width=3), marker=dict(size=7),
                         hovertemplate="K: %{x}<br>Validation error: %{y:.2f}%<extra></extra>"))
fig.add_trace(go.Scatter(x=[kbest], y=[best_error * 100], mode="markers", name="Best K",
                         marker=dict(color="#EF4444", size=13, symbol="star"),
                         hovertemplate="Best K: %{x}<br>Validation error: %{y:.2f}%<extra></extra>"))
style_figure(fig, "KNN Validation Error by Number of Neighbors", "Number of Neighbors (K)", "Average Validation Error (%)")
fig.update_xaxes(dtick=1)
fig.show()
print(f"Best K is {kbest} with average validation error {best_error * 100:.2f}%")

Best K is 5 with average validation error 20.77%


Training final knn model for k = 5

In [28]:
# remove pandas indexing,by converting to array
X_train = np.asarray(X_train)
y_train = np.asarray(y_train)

y_pred_knn = knn_algorithm(X_train, y_train, X_test, 4)

test_accuracy = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy of KNN: {test_accuracy * 100 :.2f} %")

Accuracy of KNN: 84.44 %


In [29]:
def plot_confusion_matrix_plotly(y_true, y_pred, labels, title, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    row_totals = cm.sum(axis=1, keepdims=True)
    cm_display = np.divide(cm, row_totals, out=np.zeros_like(cm, dtype=float), where=row_totals != 0) if normalize else cm
    text = np.vectorize(lambda x: f"{x:.1%}")(cm_display) if normalize else cm.astype(str)
    colorbar_title = "Proportion" if normalize else "Count"
    label_names = [str(label) for label in labels]
    fig = go.Figure(data=go.Heatmap(z=cm_display, x=label_names, y=label_names,
                                    colorscale="Blues", text=text, texttemplate="%{text}",
                                    hovertemplate="Actual: %{y}<br>Predicted: %{x}<br>Value: %{text}<extra></extra>",
                                    colorbar=dict(title=colorbar_title)))
    style_figure(fig, title, "Predicted Class", "Actual Class", height=500)
    fig.update_yaxes(autorange="reversed")
    fig.show()

In [30]:
plot_confusion_matrix_plotly(y_test, y_pred_knn, [0, 1], "KNN Confusion Matrix")

***Evaluating KNN Results***

Since we're dealing with predictions of heart disease (Medical Diagnosis), high **Recall(sensitivity)** is vital when false negatives are costly, meaning if a person is incorrectly classified as healthy. Aiming for a **Recall as close to 1.0 (100%) as possible, accepting that more false positives will appear that require further testing.** While Accuracy is less informative in medical diagnosis because it does not distinguish between the two types of errors.

This model is better at identifying the negative class than the positive, meaning it's substantially more effective at identifying the absence of heart disease than its presence.

**It achieves a High Precision (accuracy of positive prediction)**

**Overall, the CM suggests Low Recall, High Precision: The model is very accurate when it makes a positive prediction, but it misses many actual positive cases entirely.**

Indicating that an alternative classification methods may improve its ability to correctly identify positive cases.


### Part 5: Naive Bayes using Gaussian Multivariate distribution ###


Since Naive Bayes is designed to apply Bayes theorem to continuous data by assuming that the features follow a Gaussian (normal) distribution and are conditionally independent of each other

 the objective is to determine the class C that maximizes the posterior probability Given an observation (x),

**Dealing with categorical/ numerical data separately**

Since the Data is categorical, 2 things will be used: Bernoulli Naive Bayes as a classifier used for binary features, meaning for categorical columns encoded as one hot vectors and Multivariate Gaussian that handles continuous features.

Note: the target column will not be converted to binary, this classification will be treated as a multiclass classification

In [31]:
# For Gaussian part
def mean_and_cov_matrix(X,y, classes):
    # storing means and standart deviations for each class
    means = {}
    cov_matrices = {}
    priors = {}

    for element in classes:
        filtered_rows = X[y == element]
        means[element] = filtered_rows.mean(axis=0)
        variances = np.var(filtered_rows, axis=0)  # Variances for diagonal covariance
        cov_matrices[element] = np.diag(variances) # calculating diagonal covariance matrix
        # calculate initial probability (num_of-rows of a specific class/ num of all rows)
        priors[element] = filtered_rows.shape[0] / X.shape[0]

    return means, cov_matrices, priors

#evaluating the multivariate Gaussian distribution with mean vector µ and covariance matrix for a given data point x.
def mult_gauss_distribution(x, mean, cov):
    d = len(mean)  # Dimensionality of the data
    determinant = np.linalg.det(cov)  # Determinant of the covariance matrix
    fraction = 1 / ((2 * np.pi) ** (d / 2) * np.sqrt(determinant))
    inverse = np.linalg.inv(cov)  # Inverse of the covariance matrix
    exponent = -0.5 * (x - mean).T @ inverse @ (x - mean)
    return fraction * np.exp(exponent)


def probability(X_categorical, X_continuous, classes, priors, means, cov_matrices):
    # Get probabilities from BernoulliNB (categorical)
    bernoulli_probs = bernoulli_nb.predict_proba(X_categorical)

    # Get probabilities from Gaussian for continuous variables
    gaussian_probs = []
    for x in X_continuous:
        class_probs = []
        for element in classes:
            likelihood = mult_gauss_distribution(x, means[element], cov_matrices[element])
            prob = likelihood * priors[element]
            class_probs.append(prob)
        gaussian_probs.append(class_probs)

    # numpy conversuion
    gaussian_probs = np.array(gaussian_probs)

    # Combine probabilities
    combined_probs = bernoulli_probs * gaussian_probs

    # Normalize by dividing by the sum of probabilities across all classes
    combined_probs /= combined_probs.sum(axis=1, keepdims=True)

    # Return the class with the highest probability
    return np.argmax(combined_probs, axis=1)

In [32]:
# Load dataset
nb_df = heart_disease_df

# Split dataset into features and targetss
X = getDfnoTargets(nb_df)
original_dataset_cl = drop_NAN(original_dataset)
y = getTargets(original_dataset_cl)

categorical_columns = ['restecg', 'slope', 'cp', 'thal']
numerical_columns = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_cat_train = X_train.loc[:, categorical_columns].values
X_cont_train = X_train.loc[:, numerical_columns].values
X_cat_test = X_test.loc[:, categorical_columns].values
X_cont_test = X_test.loc[:, numerical_columns].values

bernoulli_nb = BernoulliNB()
bernoulli_nb.fit(X_cat_train, y_train)

BernoulliNB()

In [33]:
classes = np.unique(y_train)
#train model
means, cov_matrices, priors = mean_and_cov_matrix(X_cont_train, y_train, classes)
y_pred = probability(X_cat_test, X_cont_test, classes, priors, means, cov_matrices)

# accuracy score
test_accuracy = np.mean(y_pred == y_test)
print(f"Accuracy of Naive Bayes: {accuracy * 100 :.2f} %")

Accuracy of Naive Bayes: 63.64 %


In [34]:
plot_confusion_matrix_plotly(y_test, y_pred, classes, "Hybrid Naive Bayes Confusion Matrix")

***Evaluating Hybrid Naive Bayes Results***

The matrix displays a strong bias toward predicting class 0. Generally, the dataset is imbalanced as the 0 values prevail in the target colmun. Consequently, the classifier achieves high specificity, as it identifies majority of true negatives. However, accuracy is not an appropriate evaluation metric, as it is inflated by the majority class. As stated before, for medical diagnosis, Recall(sensitivity) will be considered and it's low value indicates that a substantial proportion of patients with heart disease remain undetected.

### Part 6: Multilayer Perceptron ###

Implementation of Multilayer Perceptron (neural network with multiple linear layers combined with non-linear activation functions) using the Pytorch framework

**General structure of MLP:**

It consists of 3 layers and contains a single hidden layer, the 1) Input layer 2) Hidden Layer 3) Output Layer

**Forward Pass:**

Note: Each perceptron will use a hyperplane using the sigmoid function(because of multidimensional dataframe), instead of a linear function as for 2D space.

$$
z = \mathbf{W}^{T}\mathbf{x} + b
$$

Afterwards, z is passed through a sigmoid function, producing a hidden layer output

W stands for weights connecting the input/output layer to the hidden layer and vice versa.
It is a matrix that tells the neural network how strongly each input feature influences each hidden neuron.

Then weights and biases of the MLP are updated using Backpropagation after each forward pass update.
First, the model computes the prediction output using forward propagation. The prediction is then compared with the true label y using binary cross-entropy loss.

Note: since the output layer uses a sigmoid activation together with binary cross-entropy loss, the error at the output layer is simplified to: **output - y** (how far the predicted probability is from the real target value)

Then the gradients for the weights between the hidden layer and output layer are calculated (how much each hidden neuron contributed to the final prediction error)

In [35]:
from sklearn.preprocessing import StandardScaler

normalized_df = heart_disease_df

X = getDfnoTargets(normalized_df)
y = getTargets(normalized_df)

X_train, X_test, y_train, y_test = split_data(X, y)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test = torch.tensor(X_test.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# Whid -> Weights computed between input layer and hidden layer
# Wout -> Weights computed between hidden layer and output layer

class MLP:
    def __init__(self, input_size, hidden_size, output_size):
        self.Whid = (0.01 * torch.randn(input_size, hidden_size)).requires_grad_(False)# initializing Weights Matrix
        self.bias_hid = torch.zeros(1, hidden_size)# bias term
        self.Wout = (0.01 * torch.randn(hidden_size, output_size)).requires_grad_(False)
        self.bias_out = torch.zeros(1, output_size)

    def forward_propagation(self, X):
        self.z1 = X @ self.Whid + self.bias_hid
        self.a1 = torch.sigmoid(self.z1)

        self.z2 = self.a1 @ self.Wout + self.bias_out
        self.a2 = torch.sigmoid(self.z2)
        return self.a2

    def bce_loss(self, y_pred, y_true, eps=1e-7):
        # touch up
        y_pred = torch.clamp(y_pred, eps, 1 - eps)
        return -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred)).mean()

    def backward_propagation(self, X, y, output, learning_r=0.01):
        m = X.shape[0]

        # For sigmoid + BCE: dz2 = a2 - y
        dz2 = (output - y)
        dWout = (self.a1.T @ dz2) / m # matrix that calculates how much each perceptron contributed to the error
        dbias_out = dz2.sum(dim=0, keepdim=True) / m
        da1 = dz2 @ self.Wout.T
        dz1 = da1 * (self.a1 * (1 - self.a1)) # derivative of sigmoid
        dWhid = (X.T @ dz1) / m
        dbias_hid = dz1.sum(dim=0, keepdim=True) / m

        # Gradient descent update
        self.Whid -= learning_r * dWhid
        self.bias_hid -= learning_r * dbias_hid
        self.Wout -= learning_r * dWout
        self.bias_out -= learning_r * dbias_out

    def predict_proba(self, X):
        return self.forward_propagation(X)

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).float()

    # using 1000 epochs when training
    def train(self, X, y, epochs=1000, learning_r=0.01):
        losses = []
        for epoch in range(epochs):
            output = self.forward_propagation(X)
            loss = self.bce_loss(output, y)
            losses.append(loss.item())

            self.backward_propagation(X, y, output, learning_r)

            if (epoch + 1) % 100 == 0:
                preds = (output >= 0.5).float()
                acc = (preds == y).float().mean().item()
                #print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss.item():.4f} Acc: {acc:.3f}")

        return losses

**Constructing and Evaluating Model**

In [36]:
# make the model
model = MLP(input_size=X_train.shape[1], hidden_size=32, output_size=1)
mlp_losses  = model.train(X_train, y_train, epochs=2000, learning_r=0.1)

# Evaluate
with torch.no_grad():
    # train acc
    train_probs = model.predict_proba(X_train)
    train_preds = (train_probs >= 0.5).float()
    train_acc = (train_preds == y_train).float().mean().item()

    #test acc
    test_probs = model.predict_proba(X_test)
    test_preds = (test_probs >= 0.5).float()
    test_acc = (test_preds == y_test).float().mean().item()
    print("Total accuracy of Test set:", test_acc)
    print("Total accuracy of Train set:", train_acc)

Total accuracy of Test set: 0.9666666388511658
Total accuracy of Train set: 1.0


In [37]:
plot_learning_curve(mlp_losses)

***Evaluating MLP***

The results display that MLP converged successfully, shown by the continuous **decrease in cross-entropy loss** during training. Moreover, it achieved **Accuracy of 100% on the training set** and **96.67% on the test set** and thus demonstrates high predictive performance and good generalization. However, there are signs of overfitting due to the perfect training accuracy.

### Part 8: K-Means Algorithm ###

Since the dataset consists of Mixed data, K-medoids will be used instead of classic K-means. Kmeans is a centroid method where each cluster is represented by the mean of its points. Whereas, Kmedoids uses an an actual data point within the cluster whose total distance to all other points in the same cluster is minimal.

The  dataset is already normalized and each categorical column is already represented using one hot vectors, pairwise Manhattan distance will be used, specifically pairwise distance matrix.

The goal is to find the cluster assignments that minimize the total distance between each data point and its assigned medoid.

<br>***The objective function is :***</br>

$$
\underset{\{C_1,\ldots,C_k\}}{\operatorname{argmin}}
\sum_{h=1}^{k}
\sum_{x_i \in C_h}
d(x_i, m_h)
$$

The algorithm will be evaluated based on its convergence, meaning how many steps the k-medoids algorithm needed before it stopped improving.
The algorithm tries all possible swaps, by replacing the current medoid with a non-medoid point and computes whether this swap reduces the total cluster cost (sum of distances within cluster)

In [38]:
from typing import Tuple

def k_medoids(D, k, max_iter=100, random_seed=42):
    np.random.seed(random_seed)
    N = D.shape[0]
    all_idxs = np.arange(N)

#   creating random medoid indexes
    medoids_idx = np.random.choice(N, k, replace=False)

# initialize distances
    distances = np.zeros((N, k))
    medoid_history = [medoids_idx.copy()]

# D :is the pairwise manhattan distance matrix, symmetric
# ditances: matrix that has data points as rows, and cluster medoids as columns. It computes distances from every data point to the medoids


    for i in range(k):
        distances[:, i] = D[:, medoids_idx[i]]# D[:, i] -> take all the rows from colum i, these are the distances to medoid i.

    # select min distance, collect points to a cluster i
    #argmin instead of min, so it returns the index of the minimum
    labels = np.argmin(distances, axis=1)# axis=1 ->Look across each row

    for it in range(max_iter):
        best_swap = (-1, -1, 0.0) # (i->cluster index, j->point in cluster, cost_change)
        best_column = None

#       collect indexes all points that are not medoids themselves
        non_medoids_idx = all_idxs[~np.isin(all_idxs, medoids_idx)]

        for i in range(k):
#           boolean mask(array) has True/False values used to select or filter elements from another array
#           collects all points that belong to the same cluster i.
            cluster_mask = (labels == i)

            if not np.any(cluster_mask): # if there are no points in curretn cluster skip the cluster
                continue

            current_cost = distances[cluster_mask, i].sum()# Boolean indexing, e.g Row 0 -> False -> Ignore

            # swapping medoids and non medoids
            for j in non_medoids_idx:

                new_col = D[:, j]# take column at j
#                 print(new_col)
                # compute distance from every point to point j
                new_cost = new_col[cluster_mask].sum()
                cost_change = new_cost - current_cost

                if cost_change < best_swap[2]:
                    best_swap = (i, j, float(cost_change))
                    best_column = new_col

        if best_swap[0] == -1:
            break

        i, j, _ = best_swap
        medoids_idx[i] = j
        distances[:, i] = best_column
        labels = np.argmin(distances, axis=1)
        medoid_history.append(medoids_idx.copy())

    print( f"K-medoids converged after {it + 1} iterations")
    return labels, medoids_idx, np.array(medoid_history)

# returns a distance matrix
def manhattan(X: np.ndarray) -> np.ndarray:
    # X: (N, D)
    X = X.astype(np.float64)
    Dist_matrix = np.abs(X[:, None, :] - X[None, :, :]).sum(axis=2)
    return Dist_matrix

***Finding the optimal value of hyperparameter k***

Silhouette coefficient: provides a score that grades the cluster assignment of a data point, in order to obtain a single score for the whole partition of dataset

- +1 means the point is ideally clustered — it’s much closer to points in its own cluster than to points in the nearest cluster.
- 0 suggests the point is on the boundary between clusters, not really belonging to any particular group.
- -1 means it’s been misclassified — it’s closer to another cluster than its own.

Micro-Silhouette score: silhouette averaged at the data point level, equal weight is assigned to every point.

Macro-Silhouette score: silhouette averaged at the cluster level, equal weight to every cluster.

In [39]:
def silhouette(labels, D):
    labels = np.asarray(labels)
    unique_clusters = np.unique(labels)


    if len(unique_clusters) < 2:
        return np.array([0] * len(labels))

    silhouette_values = []

    for i in range(len(labels)):
        # current cluster number
        cluster_points = np.where(labels == labels[i])[0]# collects indexes of points

        # remove i from pointss
        cluster_points = cluster_points[cluster_points != i]

        # b_i: minimum average distance to another cluster
        b_i = np.inf

        # a_i: average distance to points in same cluster
        if len(cluster_points) == 0:
            a_i = 0
        else:
            a_i = np.mean(D[i, cluster_points])# used instead of D[i][same_cluster], because the second element is an array

        for cluster in unique_clusters:
            if cluster == labels[i]:
                continue

            other_cluster = np.where(labels == cluster)[0]

            avg_distance = np.mean(D[i, other_cluster])

            if avg_distance < b_i:
                b_i = avg_distance

        # silhouette value for point i
        if max(a_i, b_i) == 0:
            s_i = 0

        else:
            s_i = (b_i - a_i) / max(a_i, b_i)

        silhouette_values.append(s_i)

    return silhouette_values

def micro_silhouette_score(labels, silhouette_values):
    return np.mean(silhouette_values)

def macro_silhouette_score(labels, silhouette_values):
    labels = np.asarray(labels)
    silhouette_values = np.asarray(silhouette_values)

    cluster_scores = []

    for cluster in np.unique(labels):
        cluster_mean = np.mean(silhouette_values[labels == cluster])
        cluster_scores.append(cluster_mean)

    return np.mean(cluster_scores)

In [40]:
X_kmeans = np.asarray(X_train)

D = manhattan(X_kmeans)

micro_scores = []
macro_scores = []
for k in range(2,20):
    labels, medoids_idx, history = k_medoids(D, k, max_iter=50)
    silhouette_values = silhouette(labels, D)
    micro = micro_silhouette_score(labels, silhouette_values)
    macro = macro_silhouette_score(labels, silhouette_values)
    micro_scores.append(micro)
    macro_scores.append(macro)
#     print(f"For k: {k} the micro is {micro} and macro is {macro}")

K-medoids converged after 4 iterations
K-medoids converged after 3 iterations
K-medoids converged after 5 iterations
K-medoids converged after 4 iterations
K-medoids converged after 9 iterations
K-medoids converged after 7 iterations
K-medoids converged after 10 iterations
K-medoids converged after 8 iterations
K-medoids converged after 10 iterations
K-medoids converged after 13 iterations
K-medoids converged after 10 iterations
K-medoids converged after 10 iterations
K-medoids converged after 10 iterations
K-medoids converged after 11 iterations
K-medoids converged after 13 iterations
K-medoids converged after 11 iterations
K-medoids converged after 11 iterations
K-medoids converged after 11 iterations


In [41]:
k_values = np.arange(2, 20)
fig = go.Figure()
fig.add_trace(go.Scatter(x=k_values, y=macro_scores, mode="lines+markers",
                         name="Macro score", line=dict(color="#2563EB", width=3), marker=dict(size=7),
                         hovertemplate="Clusters: %{x}<br>Macro score: %{y:.4f}<extra></extra>"))
fig.add_trace(go.Scatter(x=k_values, y=micro_scores, mode="lines+markers",
                         name="Micro score", line=dict(color="#F97316", width=3),
                         marker=dict(size=7), hovertemplate="Clusters: %{x}<br>Micro score: %{y:.4f}<extra></extra>"))
style_figure(fig, "Silhouette Scores by Number of Clusters", "Number of Clusters (K)", "Silhouette Score")
fig.update_xaxes(dtick=1)
fig.show()

***Evaluating K-medoids***

The silhouette analysis indicates that at **k=2 both micro and macro silhouette scores are the highest.**
<br>suggesting that it is the most appropriate number of clusters in this case. </br>
In general the silhouette score remains below 0.30 for all values of k, indicating weak cluster separation and limited cohesion within the dataset. Although the micro silhouette score increases for larger values of k, the corresponding macro silhouette score decreases, suggesting that the improvement is not consistent across all clusters. Overall, the results imply that the dataset does not exhibit a well-defined clustering structure, and the quality of the obtained partitions remains poor regardless of the selected number of clusters.